In [92]:
print('Hello World!!')

Hello World!!


In [93]:
#Creating a Spark Session
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("movies") \
    .getOrCreate()

In [94]:
#Reading the datasets - 1) Ad Campaigns
from pyspark.sql.types import StructField, StructType, StringType, TimestampType
hdfs_location = '/bootcamp/spark_assgn1/datasets/ad_campaign_data.json'

ad_campaign_schema = StructType([
    StructField('campaign_country', StringType(), False), 
    StructField('campaign_id', StringType(), False), 
    StructField('campaign_name', StringType(), False), 
    StructField('device_type', StringType(), False), 
    StructField('event_time', TimestampType(), False), 
    StructField('event_type', StringType(), False), 
    StructField('os_type', StringType(), False), 
    StructField('place_id', StringType(), False), 
    StructField('user_id', StringType(), False)
])

ad_campaign_df = spark.read.option("multiLine", "true").schema(ad_campaign_schema).json(hdfs_location)
ad_campaign_df.show(truncate = False)
#ad_campaign_df.printSchema()

+----------------+-----------+-----------------------------+-----------+-------------------+----------+-------+---------+-------------------+
|campaign_country|campaign_id|campaign_name                |device_type|event_time         |event_type|os_type|place_id |user_id            |
+----------------+-----------+-----------------------------+-----------+-------------------+----------+-------+---------+-------------------+
|USA             |ABCDFAE    |Food category target campaign|apple      |2018-10-12 13:10:05|impression|ios    |CASSBB-11|1264374214654454321|
|USA             |ABCDFAE    |Food category target campaign|MOTOROLA   |2018-10-12 09:04:00|impression|android|CADGBD-13|1674374214654454321|
|USA             |ABCDFAE    |Food category target campaign|SAMSUNG    |2018-10-12 13:10:10|video ad  |android|BADGBA-12|5747421465445443   |
|USA             |ABCDFAE    |Food category target campaign|SAMSUNG    |2018-10-12 13:10:12|click     |android|CASSBB-11|1864374214654454132|
+-----

In [95]:
#Reading the datasets - 2) store_data.json
hdfs_location = '/bootcamp/spark_assgn1/datasets/store_data.json'

store_data_df = spark.read.option("multiLine", "true").json(hdfs_location)
store_data_df.show(truncate = False)

+---------------------------------+-------------+
|place_ids                        |store_name   |
+---------------------------------+-------------+
|[CASSBB-11, CADGBD-13, FDBEGD-14]|McDonald     |
|[CASSBB-11]                      |BurgerKing   |
|[CASSBB-11]                      |BurgerKing   |
|[BADGBA-13, CASSBB-15, FDBEGD-15]|Macys        |
|[BADGBA-13, CASSBB-15, FDBEGD-15]|shoppers stop|
+---------------------------------+-------------+



In [96]:
#Reading the datasets - 3) user_profile_data.json
hdfs_location = '/bootcamp/spark_assgn1/datasets/user_profile_data.json'

user_profile_df = spark.read.option("multiLine", "true").json(hdfs_location)
user_profile_df.show(truncate = False)

+---------+-------------------------------+-------+------+-------------------+
|age_group|category                       |country|gender|user_id            |
+---------+-------------------------------+-------+------+-------------------+
|18-25    |[shopper, student]             |USA    |male  |1264374214654454321|
|25-50    |[parent]                       |USA    |female|1674374214654454321|
|25-50    |[shopper, parent, professional]|USA    |male  |5747421465445443   |
|50+      |[professional]                 |USA    |male  |1864374214654454132|
|18-25    |[shopper, student]             |USA    |female|14537421465445443  |
|50+      |[shopper, professional]        |USA    |female|25547421465445443  |
+---------+-------------------------------+-------+------+-------------------+



In [97]:
# Problem Statement 1 : Q1. Analyse data for each campaign_id, date, hour, os_type & value to get all the events with counts.
from pyspark.sql.functions import col, to_date, hour, count, first, struct

ans_1 = ad_campaign_df \
        .withColumn('event_date', to_date(col('event_time'))) \
        .withColumn('event_hour', hour(col('event_time'))) \
        .groupBy(col('campaign_id'), col('event_date'), col('event_hour'), col('event_type'), col('os_type')) \
        .agg(count(col('event_type')).alias('event_count')) \
        .groupBy(col('campaign_id'), col('event_date'), col('event_hour'), col('os_type')) \
        .pivot('event_type') \
        .agg(first(col('event_count'))) \
        .fillna(0) \
        .select(
            col('campaign_id'), 
            col('event_date'), 
            col('event_hour'), 
            col('os_type'),
            struct(
                col('click').alias('click'), 
                col('impression').alias('impression'), 
                col('`video ad`').alias('video_ad') # Added backticks and renamed alias to avoid spaces
            ).alias('event') # Added the alias for the struct column
        )

ans_1.show(truncate=False)

+-----------+----------+----------+-------+---------+
|campaign_id|event_date|event_hour|os_type|event    |
+-----------+----------+----------+-------+---------+
|ABCDFAE    |2018-10-12|13        |android|{1, 0, 1}|
|ABCDFAE    |2018-10-12|13        |ios    |{0, 1, 0}|
|ABCDFAE    |2018-10-12|9         |android|{0, 1, 0}|
+-----------+----------+----------+-------+---------+



In [98]:
#Writing results to HDFS 
hdfs_output_location1 = '/bootcamp/spark_assgn1/output/result1/'

ans_1.write.mode("overwrite").json(hdfs_output_location1 + "q1_output.json")

In [99]:
# Problem Statement 2 - Analyse data for each campaign_id, date, hour, store_name & value to get all the events with counts

from pyspark.sql.functions import explode

#Exploding place_ids column in store_data_df. 
store_data_df = store_data_df.withColumn("exploded_place_ids", explode(col('place_ids')))
store_data_df.show(truncate = False)

+---------------------------------+-------------+------------------+
|place_ids                        |store_name   |exploded_place_ids|
+---------------------------------+-------------+------------------+
|[CASSBB-11, CADGBD-13, FDBEGD-14]|McDonald     |CASSBB-11         |
|[CASSBB-11, CADGBD-13, FDBEGD-14]|McDonald     |CADGBD-13         |
|[CASSBB-11, CADGBD-13, FDBEGD-14]|McDonald     |FDBEGD-14         |
|[CASSBB-11]                      |BurgerKing   |CASSBB-11         |
|[CASSBB-11]                      |BurgerKing   |CASSBB-11         |
|[BADGBA-13, CASSBB-15, FDBEGD-15]|Macys        |BADGBA-13         |
|[BADGBA-13, CASSBB-15, FDBEGD-15]|Macys        |CASSBB-15         |
|[BADGBA-13, CASSBB-15, FDBEGD-15]|Macys        |FDBEGD-15         |
|[BADGBA-13, CASSBB-15, FDBEGD-15]|shoppers stop|BADGBA-13         |
|[BADGBA-13, CASSBB-15, FDBEGD-15]|shoppers stop|CASSBB-15         |
|[BADGBA-13, CASSBB-15, FDBEGD-15]|shoppers stop|FDBEGD-15         |
+---------------------------------

In [100]:
#Joining store_data_df and ad_campaign_df

from pyspark.sql.functions import col, to_date, hour, count, first, struct

ans_2 = ad_campaign_df.alias('ad').join( 
            store_data_df.alias('stor'), 
            col('stor.exploded_place_ids') == col('ad.place_id'), 
            how = 'inner' 
        ) \
        .withColumn('event_date', to_date(col('ad.event_time'))) \
        .withColumn('event_hour', hour(col('ad.event_time'))) \
        .groupBy(col('ad.campaign_id'), col('event_date'), col('event_hour'), col('ad.event_type'), col('stor.store_name')) \
        .agg(count(col('ad.event_type')).alias('event_count')) \
        .groupBy(col('campaign_id'), col('event_date'), col('event_hour'), col('store_name')) \
        .pivot('event_type', ['click', 'impression', 'video ad']) \
        .agg(first(col('event_count'))) \
        .fillna(0) \
        .select(
            # REMOVED ad. and stor. prefixes here
            col('campaign_id'), 
            col('event_date'), 
            col('event_hour'), 
            col('store_name'), 
            struct(
                col('click').alias('click'), 
                col('impression').alias('impression'),  
                col('`video ad`').alias('video_ad') 
            ).alias('event')
        )

ans_2.show()

+-----------+----------+----------+----------+---------+
|campaign_id|event_date|event_hour|store_name|    event|
+-----------+----------+----------+----------+---------+
|    ABCDFAE|2018-10-12|         9|  McDonald|{0, 1, 0}|
|    ABCDFAE|2018-10-12|        13|  McDonald|{1, 1, 0}|
|    ABCDFAE|2018-10-12|        13|BurgerKing|{2, 2, 0}|
+-----------+----------+----------+----------+---------+



In [101]:
#Writing results to HDFS 
hdfs_output_location2 = '/bootcamp/spark_assgn1/output/result2/'

ans_2.write.mode("overwrite").json(hdfs_output_location2 + "q2_output.json")

In [102]:
# Read the data you just wrote
verify_df = spark.read.json(hdfs_output_location2 + "q2_output.json")

# Show the results
verify_df.show(truncate=False)

+-----------+---------+----------+----------+----------+
|campaign_id|event    |event_date|event_hour|store_name|
+-----------+---------+----------+----------+----------+
|ABCDFAE    |{0, 1, 0}|2018-10-12|9         |McDonald  |
|ABCDFAE    |{1, 1, 0}|2018-10-12|13        |McDonald  |
|ABCDFAE    |{2, 2, 0}|2018-10-12|13        |BurgerKing|
+-----------+---------+----------+----------+----------+



In [103]:
#Problem Statement 3 : Analyse data for each campaign_id, date, hour, gender_type & value to get all the events with counts

ans_3 = ad_campaign_df.alias('ad').join( 
            user_profile_df.alias('us'), 
            col('us.user_id') == col('ad.user_id'), 
            how = 'inner' 
        ) \
        .withColumn('event_date', to_date(col('ad.event_time'))) \
        .withColumn('event_hour', hour(col('ad.event_time'))) \
        .groupBy(col('ad.campaign_id'), col('event_date'), col('event_hour'), col('ad.event_type'), col('us.gender')) \
        .agg(count(col('ad.event_type')).alias('event_count')) \
        .groupBy(col('campaign_id'), col('event_date'), col('event_hour'), col('gender')) \
        .pivot('event_type', ['click', 'impression', 'video ad']) \
        .agg(first(col('event_count'))) \
        .fillna(0) \
        .select(
            # REMOVED ad. and stor. prefixes here
            col('campaign_id'), 
            col('event_date'), 
            col('event_hour'), 
            col('gender'), 
            struct(
                col('click').alias('click'), 
                col('impression').alias('impression'),  
                col('`video ad`').alias('video_ad') 
            ).alias('event')
        )

ans_3.show()

+-----------+----------+----------+------+---------+
|campaign_id|event_date|event_hour|gender|    event|
+-----------+----------+----------+------+---------+
|    ABCDFAE|2018-10-12|        13|  male|{1, 1, 1}|
|    ABCDFAE|2018-10-12|         9|female|{0, 1, 0}|
+-----------+----------+----------+------+---------+



In [106]:
#Writing results to HDFS 
hdfs_output_location3 = '/bootcamp/spark_assgn1/output/result3/'

ans_3.write.mode("overwrite").json(hdfs_output_location3 + "q3_output.json")